In [1]:
!pip install pdfplumber
!pip install requests
!pip install python-dotenv

In [4]:
# --- Resume Analyzer Sandbox ---
# This notebook demonstrates the core text extraction and AI scoring logic.

import os
import json
import requests
import pdfplumber
from dotenv import load_dotenv

# 1. Load Environment Variables
load_dotenv(override=True)
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise ValueError("API Key missing. Please check your .env file.")

# 2. Extract Text from PDF
file_path = "uploads/RESUME.pdf" 
text = ""

if os.path.exists(file_path):
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
else:
    print(f"Could not find {file_path}")

# 3. Call NVIDIA Gemma 31B API
if text:
    invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_API_KEY}",
        "Accept": "application/json"
    }
    
    prompt = f"""
    You are an expert ATS system and recruiter. Analyze the following resume text.
    Return ONLY a valid JSON object matching this exact structure:
    {{
      "score": 85,
      "sections": {{
        "content": {{"score": 90, "feedback": "..."}},
        "formatting": {{"score": 80, "feedback": "..."}},
        "keywords": {{"score": 75, "feedback": "..."}},
        "ats": {{"score": 95, "feedback": "..."}}
      }},
      "suggestions": [
        "Suggestion 1",
        "Suggestion 2"
      ]
    }}

    Resume Text:
    {text}
    """

    payload = {
        "model": "google/gemma-4-31b-it",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
        "temperature": 0.2,
        "stream": False
    }

    print("Analyzing...")
    response = requests.post(invoke_url, headers=headers, json=payload, timeout=60)

    if response.status_code == 200:
        ai_response_text = response.json()['choices'][0]['message']['content']
        clean_json_string = ai_response_text.strip('```json').strip('```').strip()
        final_results = json.loads(clean_json_string)
        print(json.dumps(final_results, indent=2))
    else:
        print(f"Error: {response.status_code}")

Analyzing...
Error: 502
